# Patch — Market-wide and sector/supply-chain news indices

Этот блок запускается после базового Notebook 1. Он:
1. загружает дополнительные новости из GDELT по market-wide и sector/supply-chain запросам;
2. классифицирует их той же FinBERT-моделью, что и firm-specific новости;
3. агрегирует индексы `I_market` и `I_sector`;
4. обновляет `data/final_features_daily.parquet`, не удаляя старый `I_t`.

Предполагается, что базовый ноутбук уже создал:
- `data/news_scored_all.parquet`
- `data/final_features_daily.parquet`

Если у вас есть локальная fine-tuned модель, укажите путь в `FINETUNED_MODEL_PATH`.

In [5]:

from pathlib import Path
from datetime import datetime, timedelta, timezone
import os, time, math, re, json, random, logging
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

START_DATE = pd.Timestamp("today").normalize() - pd.DateOffset(years=5)
END_DATE = pd.Timestamp("today").normalize()

# Если ваша fine-tuned модель лежит в локальной папке, укажите её здесь.
# Примеры: "ProsusAI_finbert_fine_tuned_1", "PorousAI_finbert_fine_tuned_1"
FINETUNED_MODEL_PATH = os.environ.get("FINETUNED_MODEL_PATH", "ProsusAI_finbert_fine_tuned_1")
BASE_MODEL_NAME = "ProsusAI/finbert"

print("Date range:", START_DATE.date(), "→", END_DATE.date())
print("Fine-tuned model path candidate:", FINETUNED_MODEL_PATH)


Date range: 2021-05-19 → 2026-05-19
Fine-tuned model path candidate: ProsusAI_finbert_fine_tuned_1


In [6]:

# --- Robust parquet helpers ---
def safe_read_parquet(path: Path) -> pd.DataFrame:
    path = Path(path)
    try:
        return pd.read_parquet(path)
    except Exception as e1:
        try:
            return pd.read_parquet(path, engine="fastparquet")
        except Exception as e2:
            raise RuntimeError(f"Cannot read {path}: {e1} / {e2}")

def safe_to_parquet(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
    except Exception:
        df.to_parquet(path, index=False, engine="fastparquet")

def normalize_day_utc_naive(x):
    ts = pd.to_datetime(x, utc=True, errors="coerce")
    if pd.isna(ts):
        return pd.NaT
    return ts.tz_convert("UTC").tz_localize(None).normalize()

def build_text(row):
    title = "" if pd.isna(row.get("title")) else str(row.get("title"))
    summary = "" if pd.isna(row.get("summary")) else str(row.get("summary"))
    txt = (title + ". " + summary).strip()
    return re.sub(r"\s+", " ", txt)


In [9]:

# --- News category design ---
# firm-specific индекс берём из уже существующего news_scored_all.parquet.
# Дополнительно строим market-wide и sector/supply-chain индексы.

CONTEXT_QUERIES = {
    "market": {
        "AAPL": [
            '"Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P 500" OR Nasdaq'
        ],
        "XOM": [
            '"Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P 500"'
        ],
    },
    "sector": {
        "AAPL": [
            'TSMC OR Foxconn OR semiconductor OR "chip supply" OR "iPhone production" OR "smartphone demand" OR "Apple supplier" OR "China supply chain"'
        ],
        "XOM": [
            '"crude oil" OR Brent OR WTI OR OPEC OR "oil inventories" OR "EIA petroleum" OR "natural gas" OR "refinery margins" OR "oil supply"'
        ],
    },
}

# Для GDELT желательно ограничить язык английским, чтобы FinBERT не получал неанглийский текст.
GDELT_BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

# --- GDELT retry + logging ---
# Лог пишется и в консоль, и в файл outputs/gdelt_fetch.log.
# Это помогает понять, какой именно запрос/месяц упал, сколько было попыток,
# и сколько секунд код ждал перед повтором.
GDELT_LOG_PATH = OUT_DIR / "gdelt_fetch.log"

logger = logging.getLogger("gdelt_fetch")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_log_formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
_stream_handler = logging.StreamHandler()
_stream_handler.setFormatter(_log_formatter)
_file_handler = logging.FileHandler(GDELT_LOG_PATH, encoding="utf-8")
_file_handler.setFormatter(_log_formatter)

logger.addHandler(_stream_handler)
logger.addHandler(_file_handler)
logger.propagate = False


def _short_query(query: str, max_len: int = 90) -> str:
    query = re.sub(r"\s+", " ", str(query)).strip()
    return query if len(query) <= max_len else query[:max_len] + "..."


def _retry_wait_seconds(response=None, attempt: int = 1, base_wait: float = 10.0, max_wait: float = 120.0) -> float:
    """Use Retry-After when available; otherwise exponential backoff + jitter."""
    retry_after = None
    if response is not None:
        retry_after_header = response.headers.get("Retry-After")
        if retry_after_header:
            try:
                retry_after = float(retry_after_header)
            except ValueError:
                retry_after = None

    exponential_wait = min(max_wait, base_wait * (2 ** (attempt - 1)))
    jitter = random.uniform(0, min(3.0, exponential_wait * 0.2))
    wait = exponential_wait + jitter

    if retry_after is not None:
        wait = max(wait, retry_after)

    return min(wait, max_wait)


def gdelt_fetch_query(
    query: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    maxrecords: int = 250,
    max_retries: int = 5,
    timeout: int = 60,
    base_wait: float = 10.0,
    max_wait: float = 120.0,
) -> pd.DataFrame:
    """
    Fetch one GDELT ArtList query for a date interval.

    Robustness:
    - retries temporary HTTP errors: 429, 500, 502, 503, 504;
    - retries network errors like ConnectionResetError / ConnectionError / Timeout;
    - logs every attempt and final failure to outputs/gdelt_fetch.log;
    - returns an empty DataFrame instead of stopping the notebook.
    """
    params = {
        "query": f"({query}) sourcelang:english",
        "mode": "ArtList",
        "format": "json",
        "sort": "HybridRel",
        "maxrecords": maxrecords,
        "startdatetime": pd.Timestamp(start).strftime("%Y%m%d%H%M%S"),
        "enddatetime": pd.Timestamp(end).strftime("%Y%m%d%H%M%S"),
    }

    start_label = pd.Timestamp(start).strftime("%Y-%m-%d")
    end_label = pd.Timestamp(end).strftime("%Y-%m-%d")
    query_label = _short_query(query)
    retry_statuses = {429, 500, 502, 503, 504}

    for attempt in range(1, max_retries + 1):
        try:
            logger.info(
                "GDELT request attempt %s/%s | %s → %s | query=%s",
                attempt,
                max_retries,
                start_label,
                end_label,
                query_label,
            )
            r = requests.get(GDELT_BASE_URL, params=params, timeout=timeout)

            if r.status_code == 200:
                try:
                    data = r.json()
                except ValueError:
                    logger.warning(
                        "GDELT non-json response | %s → %s | query=%s | body=%s",
                        start_label,
                        end_label,
                        query_label,
                        r.text[:300],
                    )
                    return pd.DataFrame()

                arts = data.get("articles", []) or []
                rows = []
                for a in arts:
                    rows.append({
                        "published_at": a.get("seendate"),
                        "title": a.get("title"),
                        "summary": a.get("snippet"),
                        "url": a.get("url"),
                        "source": a.get("domain"),
                    })

                logger.info(
                    "GDELT success | rows=%s | %s → %s | query=%s",
                    len(rows),
                    start_label,
                    end_label,
                    query_label,
                )
                return pd.DataFrame(rows)

            if r.status_code in retry_statuses and attempt < max_retries:
                wait = _retry_wait_seconds(r, attempt=attempt, base_wait=base_wait, max_wait=max_wait)
                logger.warning(
                    "GDELT temporary HTTP %s | retry in %.1fs | %s → %s | query=%s | body=%s",
                    r.status_code,
                    wait,
                    start_label,
                    end_label,
                    query_label,
                    r.text[:300],
                )
                time.sleep(wait)
                continue

            logger.error(
                "GDELT failed HTTP %s | no more retries | %s → %s | query=%s | body=%s",
                r.status_code,
                start_label,
                end_label,
                query_label,
                r.text[:300],
            )
            return pd.DataFrame()

        except requests.exceptions.RequestException as e:
            if attempt < max_retries:
                wait = _retry_wait_seconds(None, attempt=attempt, base_wait=base_wait, max_wait=max_wait)
                logger.warning(
                    "GDELT network error | retry in %.1fs | attempt %s/%s | %s → %s | query=%s | error=%r",
                    wait,
                    attempt,
                    max_retries,
                    start_label,
                    end_label,
                    query_label,
                    e,
                )
                time.sleep(wait)
                continue

            logger.error(
                "GDELT network error | no more retries | %s → %s | query=%s | error=%r",
                start_label,
                end_label,
                query_label,
                e,
            )
            return pd.DataFrame()

def month_windows(start: pd.Timestamp, end: pd.Timestamp):
    cur = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    while cur < end:
        nxt = min(cur + pd.DateOffset(months=1), end)
        yield cur, nxt
        cur = nxt

def fetch_context_news(start=START_DATE, end=END_DATE, sleep_sec=10) -> pd.DataFrame:
    all_rows = []
    for context_type, by_ticker in CONTEXT_QUERIES.items():
        for ticker, queries in by_ticker.items():
            for query in queries:
                for a, b in tqdm(list(month_windows(start, end)), desc=f"{ticker} {context_type}", leave=False):
                    df = gdelt_fetch_query(query, a, b, maxrecords=100, timeout=300)
                    if not df.empty:
                        df["ticker"] = ticker
                        df["context_type"] = context_type
                        df["query"] = query
                        all_rows.append(df)
                    time.sleep(sleep_sec)
    if not all_rows:
        return pd.DataFrame(columns=["ticker","context_type","query","published_at","title","summary","url","source","date","text"])
    out = pd.concat(all_rows, ignore_index=True)
    out["published_at"] = pd.to_datetime(out["published_at"], utc=True, errors="coerce")
    out["date"] = out["published_at"].dt.tz_convert("UTC").dt.tz_localize(None).dt.normalize()
    out["text"] = out.apply(build_text, axis=1)
    out = out.dropna(subset=["date"])
    out = out[out["text"].str.len() >= 5].copy()
    out = out.drop_duplicates(subset=["ticker","context_type","url"])
    out = out.sort_values(["ticker","context_type","date","url"]).reset_index(drop=True)
    return out

context_raw_path = DATA_DIR / "news_context_raw.parquet"
if context_raw_path.exists():
    context_news = safe_read_parquet(context_raw_path)
    print("Loaded cached context news:", context_news.shape)
else:
    context_news = fetch_context_news()
    safe_to_parquet(context_news, context_raw_path)
    print("Saved context news:", context_news.shape)

context_news.head()


AAPL market:   0%|          | 0/60 [00:00<?, ?it/s]2026-05-19 21:41:46,226 | INFO | GDELT request attempt 1/5 | 2021-05-19 → 2021-06-19 | query="Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P ...
2026-05-19 21:44:21,637 | INFO | GDELT success | rows=100 | 2021-05-19 → 2021-06-19 | query="Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P ...
AAPL market:   2%|▏         | 1/60 [02:50<2:47:34, 170.41s/it]2026-05-19 21:44:36,642 | INFO | GDELT request attempt 1/5 | 2021-06-19 → 2021-07-19 | query="Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P ...
2026-05-19 21:45:13,145 | INFO | GDELT success | rows=100 | 2021-06-19 → 2021-07-19 | query="Federal Reserve" OR inflation OR "interest rates" OR recession OR "stock market" OR "S&P ...
AAPL market:   3%|▎         | 2/60 [03:41<1:37:07, 100.47s/it]2026-05-19 21:45:28,149 | INFO | GDELT request attempt 1/5 | 2021-07-19 → 2021-

Saved context news: (22599, 10)


,published_at,title,summary,url,source,ticker,context_type,query,date,text
0,2021-05-19 15:15:00+00:00,Why is the stock market down today ? | 5newson...,None,https://www.5newsonline.com/article/news/natio...,5newsonline.com,AAPL,market,"""Federal Reserve"" OR inflation OR ""interest ra...",2021-05-19,Why is the stock market down today ? | 5newson...
1,2021-05-19 17:00:00+00:00,More drops in Big Tech pull stocks lower ; Bit...,None,https://www.cp24.com/news/more-drops-in-big-te...,cp24.com,AAPL,market,"""Federal Reserve"" OR inflation OR ""interest ra...",2021-05-19,More drops in Big Tech pull stocks lower ; Bit...
2,2021-05-19 16:15:00+00:00,More drops in Big Tech pull stocks lower ; Bit...,None,https://www.ctvnews.ca/business/more-drops-in-...,ctvnews.ca,AAPL,market,"""Federal Reserve"" OR inflation OR ""interest ra...",2021-05-19,More drops in Big Tech pull stocks lower ; Bit...
3,2021-05-19 16:15:00+00:00,Dow Down 450 Points As Markets Head For Third ...,None,https://www.forbes.com/sites/palashghosh/2021/...,forbes.com,AAPL,market,"""Federal Reserve"" OR inflation OR ""interest ra...",2021-05-19,Dow Down 450 Points As Markets Head For Third ...
4,2021-05-19 15:15:00+00:00,Why is the stock market down today ?,None,https://www.kgw.com/article/news/nation-world/...,kgw.com,AAPL,market,"""Federal Reserve"" OR inflation OR ""interest ra...",2021-05-19,Why is the stock market down today ? .


In [10]:

# --- Load FinBERT / fine-tuned ProsusAI FinBERT ---
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name_or_path = FINETUNED_MODEL_PATH if Path(FINETUNED_MODEL_PATH).exists() else BASE_MODEL_NAME
print("Using model:", model_name_or_path)

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

id2label = {int(k): str(v).lower() for k, v in model.config.id2label.items()}
print("device:", device)
print("labels:", id2label)

def _label_id(substrs):
    for i, lab in id2label.items():
        if any(s in lab for s in substrs):
            return i
    return None

pos_id = _label_id(["positive", "pos"])
neg_id = _label_id(["negative", "neg"])
neu_id = _label_id(["neutral", "neu"])

if pos_id is None or neg_id is None:
    raise ValueError(f"Cannot infer positive/negative label ids from {id2label}")

def finbert_scores(texts, batch_size: int = 16, max_length: int = 256):
    texts = ["" if pd.isna(x) else str(x) for x in texts]
    probs_all = []
    for i in tqdm(range(0, len(texts), batch_size), desc="FinBERT", leave=False):
        batch = texts[i:i+batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True, max_length=max_length,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        probs_all.append(probs)
    return np.vstack(probs_all) if probs_all else np.empty((0, len(id2label)))


Using model: ProsusAI/finbert
device: cpu
labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [11]:

# --- Score context news ---
context_scored_path = DATA_DIR / "news_context_scored.parquet"

if context_scored_path.exists():
    context_scored = safe_read_parquet(context_scored_path)
    print("Loaded cached scored context news:", context_scored.shape)
else:
    context_scored = context_news.copy()
    probs = finbert_scores(context_scored["text"].tolist(), batch_size=16, max_length=256)
    context_scored["p_pos"] = probs[:, pos_id]
    context_scored["p_neg"] = probs[:, neg_id]
    context_scored["p_neu"] = probs[:, neu_id] if neu_id is not None else np.nan
    context_scored["s_x"] = context_scored["p_pos"] - context_scored["p_neg"]
    context_scored["abs_s_x"] = context_scored["s_x"].abs()
    context_scored["label_id"] = probs.argmax(axis=1)
    context_scored["label"] = context_scored["label_id"].map(id2label)
    context_scored["confidence"] = probs.max(axis=1)
    safe_to_parquet(context_scored, context_scored_path)
    print("Saved scored context news:", context_scored.shape)

context_scored.groupby(["ticker","context_type"]).size()


Saved scored context news: (22599, 18)


ticker  context_type
AAPL    market          5837
        sector          5541
XOM     market          5828
        sector          5393
dtype: int64

In [12]:

# --- Build firm / market / sector daily indices ---
# 1) firm-specific: old news_scored_all.parquet is interpreted as firm-specific news.
firm_path = DATA_DIR / "news_scored_all.parquet"
if not firm_path.exists():
    raise FileNotFoundError("Expected data/news_scored_all.parquet from the base Notebook 1.")

firm_news = safe_read_parquet(firm_path).copy()
firm_news["date"] = pd.to_datetime(firm_news["date"], errors="coerce").dt.normalize()
firm_news["context_type"] = "firm"
if "abs_s_x" not in firm_news.columns:
    firm_news["abs_s_x"] = firm_news["s_x"].abs()
if "confidence" not in firm_news.columns:
    prob_cols = [c for c in ["p_pos","p_neg","p_neu"] if c in firm_news.columns]
    firm_news["confidence"] = firm_news[prob_cols].max(axis=1) if prob_cols else np.nan

context_scored["date"] = pd.to_datetime(context_scored["date"], errors="coerce").dt.normalize()

all_scored_extended = pd.concat([
    firm_news[["ticker","context_type","date","text","title","url","source","p_pos","p_neg","p_neu","s_x","abs_s_x","confidence"]],
    context_scored[["ticker","context_type","date","text","title","url","source","p_pos","p_neg","p_neu","s_x","abs_s_x","confidence"]],
], ignore_index=True)

safe_to_parquet(all_scored_extended, DATA_DIR / "news_scored_all_extended_contexts.parquet")

def aggregate_daily_contexts(df: pd.DataFrame) -> pd.DataFrame:
    agg = (
        df.dropna(subset=["ticker","context_type","date","s_x"])
          .groupby(["ticker","context_type","date"], as_index=False)
          .agg(
              I=("s_x","mean"),
              I_max_abs=("abs_s_x","max"),
              n_news=("s_x","size"),
              n_strong=("abs_s_x", lambda x: int((x >= 0.5).sum())),
              p_pos_mean=("p_pos","mean"),
              p_neg_mean=("p_neg","mean"),
              confidence_mean=("confidence","mean"),
          )
    )
    wide_parts = []
    for ctx in ["firm", "market", "sector"]:
        part = agg[agg["context_type"] == ctx].drop(columns=["context_type"]).copy()
        part = part.rename(columns={
            "I": f"I_{ctx}",
            "I_max_abs": f"I_{ctx}_max_abs",
            "n_news": f"n_{ctx}",
            "n_strong": f"N_{ctx}_strong",
            "p_pos_mean": f"p_pos_{ctx}",
            "p_neg_mean": f"p_neg_{ctx}",
            "confidence_mean": f"confidence_{ctx}",
        })
        wide_parts.append(part)
    out = None
    for part in wide_parts:
        out = part if out is None else out.merge(part, on=["ticker","date"], how="outer")
    return out.sort_values(["ticker","date"]).reset_index(drop=True)

daily_context = aggregate_daily_contexts(all_scored_extended)
safe_to_parquet(daily_context, DATA_DIR / "daily_context_news_indices.parquet")
daily_context.to_csv(OUT_DIR / "daily_context_news_indices.csv", index=False)

display(daily_context.head())
display(daily_context.groupby("ticker")[["I_firm","I_market","I_sector","n_firm","n_market","n_sector"]].agg(["count","mean"]))


,ticker,date,I_firm,I_firm_max_abs,n_firm,N_firm_strong,p_pos_firm,p_neg_firm,confidence_firm,I_market,...,p_pos_market,p_neg_market,confidence_market,I_sector,I_sector_max_abs,n_sector,N_sector_strong,p_pos_sector,p_neg_sector,confidence_sector
0,AAPL,2020-12-29,-0.527073,0.527073,1.0,1.0,0.214336,0.741410,0.741410,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2020-12-30,-0.122154,0.878942,2.0,2.0,0.338687,0.460842,0.778159,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAPL,2021-01-04,0.659435,0.892991,2.0,1.0,0.670445,0.011010,0.725765,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAPL,2021-01-05,-0.162034,0.510277,3.0,1.0,0.041083,0.203117,0.780388,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAPL,2021-01-06,-0.073685,0.262288,3.0,0.0,0.047714,0.121399,0.830887,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


I_firm           I_market           I_sector           n_firm  \
        count      mean    count      mean    count      mean  count   
ticker                                                                 
AAPL      831  0.053967     1180 -0.259675     1329  0.125218    831   
XOM       701  0.076307     1226 -0.235302     1286 -0.172063    701   

                 n_market          n_sector            
            mean    count     mean    count      mean  
ticker                                                 
AAPL    4.333333     1180  4.94661     1329  4.169300  
XOM     3.928673     1226  4.75367     1286  4.193624

In [ ]:

# --- Merge context indices into final daily features ---
final_path = DATA_DIR / "final_features_all_extended.parquet"
if not final_path.exists():
    raise FileNotFoundError("Expected data/final_features_all_extended.parquet from the base Notebook 1.")

final = safe_read_parquet(final_path).copy()
final["date"] = pd.to_datetime(final["date"], errors="coerce").dt.normalize()
daily_context["date"] = pd.to_datetime(daily_context["date"], errors="coerce").dt.normalize()

# Avoid duplicate columns if re-running.
drop_cols = [c for c in final.columns if re.match(r"^(I|n|N|p_pos|p_neg|confidence)_(firm|market|sector)", c)]
final_base = final.drop(columns=drop_cols, errors="ignore")

final_ext = final_base.merge(daily_context, on=["ticker","date"], how="left")

# Backward compatibility: keep old I_t, but define I_firm from it if needed.
if "I_firm" not in final_ext.columns and "I_t" in final_ext.columns:
    final_ext["I_firm"] = final_ext["I_t"]

# For no-news days, counts should be zero; indices remain NaN by default.
for c in ["n_firm","n_market","n_sector","N_firm_strong","N_market_strong","N_sector_strong"]:
    if c in final_ext.columns:
        final_ext[c] = final_ext[c].fillna(0).astype(int)

safe_to_parquet(final_ext, DATA_DIR / "final_features_daily.parquet")
safe_to_parquet(final_ext, DATA_DIR / "final_features_daily_extended_contexts.parquet")
final_ext.to_csv(OUT_DIR / "final_features_daily_extended_contexts.csv", index=False)

print("Saved extended final features:", final_ext.shape)
print([c for c in final_ext.columns if any(s in c for s in ["I_firm","I_market","I_sector","n_firm","n_market","n_sector"])])
final_ext.head()


Saved extended final features: (2512, 32)
['I_firm', 'I_firm_max_abs', 'n_firm', 'I_market', 'I_market_max_abs', 'n_market', 'I_sector', 'I_sector_max_abs', 'n_sector']


,ticker,date,returns,RSI,MACD,I_t,I_t_max,N_t_strong,n_news,I_t_pos,...,p_pos_market,p_neg_market,confidence_market,I_sector,I_sector_max_abs,n_sector,N_sector_strong,p_pos_sector,p_neg_sector,confidence_sector
0,AAPL,2020-12-29,NaN,NaN,0.000000,-0.527073,0.527073,1.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,AAPL,2020-12-30,-0.008527,0.000000,-0.089302,-0.122154,0.878942,2.0,2.0,0.634633,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
2,AAPL,2020-12-31,-0.007703,0.000000,-0.238232,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
3,AAPL,2021-01-04,-0.024719,0.000000,-0.606907,0.659435,0.892991,1.0,2.0,0.659435,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,AAPL,2021-01-05,0.012364,8.684289,-0.764590,-0.162034,0.510277,1.0,3.0,0.060650,...,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN


## Интерпретация новых колонок

- `I_firm` — тональность company-specific новостей; совместима со старым `I_t`.
- `I_market` — общерыночный фон: ставки, инфляция, ФРС, recession, S&P 500/Nasdaq.
- `I_sector` — отраслевой/supply-chain фон: для AAPL — semiconductor/TSMC/Foxconn/iPhone production; для XOM — oil/OPEC/WTI/Brent/EIA inventories/natural gas.
- `n_*` — число новостей соответствующего уровня в торговый день.
- `I_*_max_abs` — самый сильный по модулю новостной сигнал дня.

В тексте НИР старый `I_t` описан как `I^{firm}_{i,t}`, а новые индексы как `I^{market}_{t}` и `I^{sector}_{i,t}`.